# 06 · Longitudinal coded records (paper §4.2)

§4.2 of the paper evaluates on sequences of ICD-coded encounters (CKD / COPD / diabetes).
Those cohorts are not available here, so this notebook uses a **synthetic EHR** with the
same structure and a known data-generating process (`sat/data/simulate_ehr.py`):
diagnosis codes, medications, labs with values and noise codes over 5 years of visits,
and four outcomes after the index date — CKD progression, diabetic complication, COPD
hospitalisation, death (semi-competing).

The claim tested: the transformer can **use the sequence** (order, trends, recency),
not only a static summary. Every model sees the same patients in three ways:

| representation | what the model is given |
|---|---|
| `sequence` | static tokens + the time-stamped history (visit time and lab values via the numeric-value embedding, §2.2) |
| `bag` | classic summary table: count per code, last value per lab, number of visits |
| `static` | age and sex only |

Three cohorts vary how much risk only the ordered history reveals (`temporal_signal`
0 / 0.5 / 0.9), so the result is a controlled answer, not an anecdote.

**Real data**: export any EHR (e.g. the §4.2 cohorts, MIMIC) to `data/<name>/patients.csv`
+ `events.csv` (format in `sat/data/dataset/parse_sequence.py`), run
`python -m sat.data.dataset.parse_sequence data/<name> --num-events K`, and add
`<name>` to `REAL_SOURCES`.

## Options

In [ ]:
NAME = "06_sequential_ehr"

# Synthetic longitudinal EHR cohorts (sat/data/simulate_ehr.py). temporal_signal = share
# of each outcome's risk that only the ORDERED history reveals (eGFR slope, HbA1c drift,
# recent exacerbations) vs current levels a summary table also sees.
SOURCES = {"ehrsim_static": "static", "ehrsim_base": "base", "ehrsim_temporal": "temporal"}
#   ehrsim_static   temporal_signal 0.0  -> the sequence should NOT beat the bag
#   ehrsim_base     temporal_signal 0.5
#   ehrsim_temporal temporal_signal 0.9  -> the sequence should beat the bag clearly
REAL_SOURCES = []        # e.g. ["ckd_cohort"]: data/ckd_cohort/{patients,events}.csv + meta.json
                         # (python -m sat.data.dataset.parse_sequence data/ckd_cohort --num-events K)
MAX_TOKENS = 256         # most recent tokens of history kept per patient
SEEDS      = [0, 1, 2]

REPRS   = ["sequence", "bag", "static"]
RECIPES = ["nllpch", "nllpch_sample_event_ranking"]
SEQUENCE_BASELINES = ["deephit_paper", "mensa_paper"]   # same transformer over the sequence
RUN_LLM_ON_SEQUENCE = False                             # §3 model on the histories (slower)

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/new/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Cohorts

In [ ]:
from scripts.runner import ehr_overrides
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    SOURCES = dict(list(SOURCES.items())[:1])
for src, scen in SOURCES.items():
    if not (REPO / "data" / src / "meta.json").is_file():
        cmd = [sys.executable, "-m", "sat.data.simulate_ehr", "--scenario", scen, "--out", f"data/{src}",
               "--max-tokens", str(MAX_TOKENS)] + (["--set", "n=1000"] if SMOKE_TEST else [])
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)
ALL = list(SOURCES) + list(REAL_SOURCES)
for src in ALL:
    m = json.loads((REPO / "data" / src / "meta.json").read_text())
    print(src, {k: m.get(k) for k in ["n", "event_rates", "codes_per_patient", "num_features", "prevalence"]})
    for rep in REPRS:
        r.prepare("ehrseq/survival", ehr_overrides(src, rep, REPO), label=f"{src}_{rep}")

## Plan and run

In [ ]:
def has_truth6(src):
    return (REPO / "data" / src / "truth.npz").is_file()

runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for src in ALL:
        base = {"dataset": src, "seed": s}
        for rep in REPRS:
            ov, hub = ehr_overrides(src, rep, REPO), f"{src}_{rep}"
            info = base | {"repr": rep}
            for rec in RECIPES:
                runs.append(Run(f"{hub}__ours_{rec}__s{s}", "finetune", "ehrseq/survival",
                                ov + so + [f"tasks/losses={rec}"], hub, info | {"model": "ours", "recipe": rec}))
            if rep == "sequence":
                for m in SEQUENCE_BASELINES:
                    runs.append(Run(f"{hub}__{m}__s{s}", "finetune", f"ehrseq/{m}", ov + so, hub, info | {"model": m}))
                if RUN_LLM_ON_SEQUENCE:
                    runs.append(Run(f"{hub}__llm_scratch__s{s}", "llm", "ehrseq/survival",
                                    ov + so + ["llm_init=scratch", "llm_learning_rate=5e-4"], hub, info | {"model": "llm_scratch"}))
                if has_truth6(src):  # the oracle does not depend on the representation
                    runs.append(Run(f"{src}__oracle__s{s}", "oracle", "ehrseq/survival", ov + so, hub,
                                    base | {"model": "oracle"}))
            if rep == "bag":  # a linear model needs a fixed design: the summary table
                runs.append(Run(f"{hub}__coxph__s{s}", "coxph", "ehrseq/coxph", ov + so, hub, info | {"model": "coxph"}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
* `table_<cohort>`: every model; the label says which input it got `[sequence|bag|static]`.
* `representation_paired`: sequence and static vs bag for our model, paired on seed. Expected: no gain for `ehrsim_static`, a clear gain for `ehrsim_temporal`.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)